In [31]:
import pandas as pd
import os
import glob
import re
# MY MAIN FOLDER
folder_path = "/home/spoorthy/Downloads/Cleaning_of_Data_and_Merging_into_single_excel/Payout_Summary_and_Order_Level_Sales"
excel_files = glob.glob(os.path.join(folder_path, "*.xlsx"))
summary_data = []
for file in excel_files:
    try:
        df = pd.read_excel(file, sheet_name=0, header=None)

        # Extracting the brand name from cell B5 (index 4,1 in pandas)
        brand_name = df.iloc[4, 1] if df.shape[0] > 4 and df.shape[1] > 1 else "NotFoundInDoc"    
        text_blob = "\n".join(df.astype(str).fillna("").values.flatten())
        payout_period_match = re.search(r"Payout Period\s+([0-9A-Za-z\s\-]+)", text_blob)
        payout_period = payout_period_match.group(1).strip() if payout_period_match else ""
        payout_date_match = re.search(r"Payout Settlement Date\s+([0-9A-Za-z\s]+)", text_blob)
        payout_date = payout_date_match.group(1).strip() if payout_date_match else ""
        total_payout_match = re.search(r"Total Payout\s+₹?([\d,]+\.\d+)", text_blob)
        total_payout = float(total_payout_match.group(1).replace(",", "")) if total_payout_match else 0.0
        orders_match = re.search(r"Total Orders\s*\(Delivered \+ Cancelled\)\s*(\d+)", text_blob)
        total_orders = int(orders_match.group(1)) if orders_match else 0
        utr_match = re.search(r"Bank UTR\s*([A-Z0-9]+)", text_blob)
        utr = utr_match.group(1).strip() if utr_match else ""
        summary_data.append({
            "File": os.path.basename(file),
            "Brand Name": brand_name,
            "Payout Period": payout_period,
            "Payout Date": payout_date,
            "Total Payout": total_payout,
            "Total Orders": total_orders,
            "UTR": utr
        })
    except Exception as e:
        print(f"Error processing {file}: {e}")
summary_df = pd.DataFrame(summary_data)
# Removing 'nan' values
summary_df['Payout Period'] = summary_df['Payout Period'].replace({r'\n': '', 'nan': ''}, regex=True).str.strip()
summary_df['Payout Date'] = summary_df['Payout Date'].replace({r'\n': '', 'nan': ''}, regex=True).str.strip()
summary_df_clean = summary_df.dropna(subset=['Payout Period', 'Payout Date'])
summary_df_clean.head()

,File,Brand Name,Payout Period,Payout Date,Total Payout,Total Orders,UTR
0,invoice_Annexure_444423_09042025_1744195095444...,Uruvalu Biryani,01 April - 05 AprilPayout Settlement Date08 Ap...,08 AprilTotal Payout,2807.70,16,AXISCN0957860635
1,invoice_Annexure_180801_09042025_1744199507239...,Roj Ka Khana(Daily Meals),01 April - 05 AprilPayout Settlement Date08 Ap...,08 AprilTotal Payout,26087.91,124,AXISCN0957908193
2,invoice_Annexure_4780_09042025_1744206715940.xlsx,Dilli Darbar,01 April - 05 AprilPayout Settlement Date08 Ap...,08 AprilTotal Payout,61563.00,227,AXISCN0957971597
3,invoice_Annexure_570268_09042025_1744199329588...,Koolerz,01 April - 05 AprilPayout Settlement Date08 Ap...,08 AprilTotal Payout,102.98,1,AXISCN0957907045
4,invoice_Annexure_180796_09042025_1744207214805...,Biryani Box,01 April - 05 AprilPayout Settlement Date08 Ap...,08 AprilTotal Payout,1709.95,8,AXISCN0957982367


In [32]:
import pandas as pd
import glob
import os
folder_path = "/home/spoorthy/Downloads/Cleaning_of_Data_and_Merging_into_single_excel/Payout_Summary_and_Order_Level_Sales"
excel_files = glob.glob(os.path.join(folder_path, "*.xlsx"))
payout_breakup_data = []
for file in excel_files:
    try:
        df = pd.read_excel(file, sheet_name=0, header=None)
        brand_name = df.iloc[4, 1] if df.shape[0] > 4 and df.shape[1] > 1 else "NotFoundInDoc"

        xls = pd.ExcelFile(file)
        target_sheet = None
        for sheet in xls.sheet_names:
            if "payout breakup" in sheet.lower():
                target_sheet = sheet
                break
        if not target_sheet:
            print(f"Not Found in file:{file}")
            continue
        raw_sheet = pd.read_excel(xls, sheet_name=target_sheet, header=None)
        header_row_idx = None
        for i, row in raw_sheet.iterrows():
            if row.astype(str).str.contains("Particulars", case=False).any():
                header_row_idx = i
                break

        if header_row_idx is None:
            print(f"No row found in sheet '{target_sheet}' of file: {file}")
            continue
        payout_df = pd.read_excel(xls, sheet_name=target_sheet, header=header_row_idx)
        payout_df = payout_df.loc[:, ~payout_df.columns.astype(str).str.contains("^Unnamed")]
        payout_df.reset_index(drop=True, inplace=True)
        payout_df["Brand Name"] = brand_name
        payout_df["Source File"] = os.path.basename(file)
        payout_breakup_data.append(payout_df)

    except Exception as e:
        print(f"Coudnt process file {file}:\n{e}")
breakup_summary_df = pd.concat(payout_breakup_data, ignore_index=True)
breakup_summary_df.head()


,Particulars,Delivered Orders,Cancelled Orders,Total,Brand Name,Source File
0,Orders,16.00,0.0,16.00,Uruvalu Biryani,invoice_Annexure_444423_09042025_1744195095444...
1,Total Customer Paid [1+2-3+4],4272.35,0.0,4272.35,Uruvalu Biryani,invoice_Annexure_444423_09042025_1744195095444...
2,Item Total,5846.00,0.0,5846.00,Uruvalu Biryani,invoice_Annexure_444423_09042025_1744195095444...
3,Packaging Charges,342.00,0.0,342.00,Uruvalu Biryani,invoice_Annexure_444423_09042025_1744195095444...
4,Discount Share,-2119.13,0.0,-2119.13,Uruvalu Biryani,invoice_Annexure_444423_09042025_1744195095444...


In [35]:
import pandas as pd
import glob
import os

folder_path = "/home/spoorthy/Downloads/Cleaning_of_Data_and_Merging_into_single_excel/Payout_Summary_and_Order_Level_Sales"
excel_files = glob.glob(os.path.join(folder_path, "*.xlsx"))
payout_breakup_data = []
for file in excel_files:
    try:
        df = pd.read_excel(file, sheet_name=0, header=None)
        brand_name = df.iloc[4, 1] if df.shape[0] > 4 and df.shape[1] > 1 else "NotFoundInDoc"
        xls = pd.ExcelFile(file)        
        order_level_data = None
        for sheet in xls.sheet_names:
            sheet_df = pd.read_excel(xls, sheet_name=sheet, header=None)
            if sheet_df.iloc[23, 1] == "Order Level":
                order_level_data = sheet_df.iloc[23:].reset_index(drop=True)
                break
        if order_level_data is None:
            print(f"No 'Order Level' data found in file: {file}")
            continue
        order_level_data = order_level_data.loc[:, ~order_level_data.columns.astype(str).str.contains("^Unnamed")]
        order_level_data.reset_index(drop=True, inplace=True)
        order_level_data["Brand Name"] = brand_name
        order_level_data["Source File"] = os.path.basename(file)
        payout_breakup_data.append(order_level_data)

    except Exception as e:
        print(f"Failed to process file {file}:\n{e}")
breakup_summary_df = pd.concat(payout_breakup_data, ignore_index=True)
breakup_summary_df.head()

,0,1,2,Brand Name,Source File
0,NaN,Order Level,"Detailed view of your order level earnings, de...",Uruvalu Biryani,invoice_Annexure_444423_09042025_1744195095444...
1,NaN,Unresolved Customer Complaints,Orders that belong to this payout period but h...,Uruvalu Biryani,invoice_Annexure_444423_09042025_1744195095444...
2,NaN,Other Charges and Deductions,Summary of all restaurant level deductions and...,Uruvalu Biryani,invoice_Annexure_444423_09042025_1744195095444...
3,NaN,Discount Summary,List of all active discounts applied on orders...,Uruvalu Biryani,invoice_Annexure_444423_09042025_1744195095444...
4,NaN,Glossary,Descriptions of all fields used in the Order L...,Uruvalu Biryani,invoice_Annexure_444423_09042025_1744195095444...


In [21]:
!pip install pymupdf



Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 8.6 MB/s eta 0:00:00m eta 0:00:010:01:01
    numpy (<2,>=1.18.*)
              ~~~~~~~^
    numpy (>=1.18.*)
           ~~~~~~~^


In [22]:
!pip install pdfplumber

Defaulting to user installation because normal site-packages is not writeable
INFO: pip is looking at multiple versions of pdfplumber to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 8.5 MB/s eta 0:00:00 MB/s eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 5.9 MB/s eta 0:00:00 MB/s eta 0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 11.9 MB/s eta 0:00:0031m12.3 MB/s eta 0:00:01
    numpy (<2,>=1.18.*)
              ~~~~~~~^
    numpy (>=1.18.*)
           ~~~~~~~^
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.12.0+cpu requires torch==1.11.0, but you have torch 1.10.1 which is incompatible.


In [36]:
import fitz
import os
import re
import pandas as pd

def extract_pdf_data(pdf_path):
    doc = fitz.open(pdf_path)
    pdf_data = {}
    page = doc[0]
    text = page.get_text("text")
    invoice_number = re.search(r"Invoice Number\s*:\s*(\S+)", text)
    if invoice_number:
        pdf_data['Invoice Number'] = invoice_number.group(1)
    gstin = re.search(r"GSTIN\s*:\s*(\S+)", text)
    if gstin:
        pdf_data['GSTIN'] = gstin.group(1)
    grand_total = re.search(r"Grand Total\s*([\d,]+\.\d+)", text)
    if grand_total:
        pdf_data['Grand Total'] = grand_total.group(1)
    service_period = re.search(r"Service Period\s*:\s*(\d{2}/\d{2}/\d{4}) to (\d{2}/\d{2}/\d{4})", text)
    if service_period:
        pdf_data['Service Period Start'] = service_period.group(1)
        pdf_data['Service Period End'] = service_period.group(2)

    return pdf_data

folder_path = "/home/spoorthy/Downloads/Cleaning_of_Data_and_Merging_into_single_excel/Commission_Invoices"
pdf_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(".pdf")]
all_pdf_data = []

for pdf_file in pdf_files:
    print(f"Extracting data from: {pdf_file}")
    pdf_data = extract_pdf_data(pdf_file)
    pdf_data['Source File'] = os.path.basename(pdf_file)
    all_pdf_data.append(pdf_data)

df = pd.DataFrame(all_pdf_data)
df.to_csv("/home/spoorthy/Downloads/Cleaning_of_Data_and_Merging_into_single_excel/Commission_Invoices/extracted_pdf_data_spoo.csv", index=False)
df.head()


Extracting data from: /home/spoorthy/Downloads/Cleaning_of_Data_and_Merging_into_single_excel/Commission_Invoices/taco_Tax_Invoice_330114_09042025_250409FS29011158.pdf
Extracting data from: /home/spoorthy/Downloads/Cleaning_of_Data_and_Merging_into_single_excel/Commission_Invoices/taco_Tax_Invoice_4780_09042025_250409FS29009158.pdf
Extracting data from: /home/spoorthy/Downloads/Cleaning_of_Data_and_Merging_into_single_excel/Commission_Invoices/taco_Tax_Invoice_313834_09042025_250409FS29011069.pdf
Extracting data from: /home/spoorthy/Downloads/Cleaning_of_Data_and_Merging_into_single_excel/Commission_Invoices/taco_Tax_Invoice_570268_09042025_250409FS29006449.pdf
Extracting data from: /home/spoorthy/Downloads/Cleaning_of_Data_and_Merging_into_single_excel/Commission_Invoices/taco_Tax_Invoice_445384_09042025_250409FS29006872.pdf
Extracting data from: /home/spoorthy/Downloads/Cleaning_of_Data_and_Merging_into_single_excel/Commission_Invoices/taco_Tax_Invoice_180801_09042025_250409FS2900723

,Invoice Number,GSTIN,Grand Total,Service Period Start,Service Period End,Source File
0,250409FS29011158,29AAFCB7707D1ZQ,"7,046.41",01/04/2025,05/04/2025,taco_Tax_Invoice_330114_09042025_250409FS29011...
1,250409FS29009158,29AAFCB7707D1ZQ,"54,571.712",01/04/2025,05/04/2025,taco_Tax_Invoice_4780_09042025_250409FS2900915...
2,250409FS29011069,29AAFCB7707D1ZQ,"6,546.279",01/04/2025,05/04/2025,taco_Tax_Invoice_313834_09042025_250409FS29011...
3,250409FS29006449,29AAFCB7707D1ZQ,122.448,01/04/2025,05/04/2025,taco_Tax_Invoice_570268_09042025_250409FS29006...
4,250409FS29006872,29AAFCB7707D1ZQ,"23,992.754",01/04/2025,05/04/2025,taco_Tax_Invoice_445384_09042025_250409FS29006...
